# Data preprocessing

Prepare the e-commerce user behavior dataset for modeling (based on `data_exploration.ipynb` findings).

In [13]:
import pandas as pd
import numpy as np
import os

import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display

## Load data

In [14]:
path = kagglehub.dataset_download("asifxzaman/e-commerce-behavior-dataset8000-users")
data = pd.read_csv(os.path.join(path, "ecommerce_user_behavior_8000.csv"))
print("Raw shape:", data.shape)

Raw shape: (8000, 14)


## 1. Working copy

In [15]:
df = data.copy()
print("Rows before cleaning:", len(df))

Rows before cleaning: 8000


## 2. Missing values

In [16]:
print("Missing per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows with missing target
df = df.dropna(subset=["purchase"])

# Drop rows with missing feature values (mainly ad_clicked)
feature_cols_raw = [c for c in df.columns if c not in ("user_id", "purchase")]
df = df.dropna(subset=feature_cols_raw)

print("Rows after dropping missing:", len(df))

Missing per column:
user_id               160
age                   160
gender                160
device_type           160
time_on_site          160
pages_viewed          160
previous_purchases    160
cart_items            160
discount_seen         160
ad_clicked            160
returning_user        160
avg_session_time      160
bounce_rate           160
purchase              160
dtype: int64
Rows after dropping missing: 6138


## 3. Target and features (drop `user_id`)

In [17]:
y = df["purchase"].astype(int)
X = df.drop(columns=["user_id", "purchase"])

print("Target distribution:")
print(y.value_counts())
print("\nFeature columns:", list(X.columns))

Target distribution:
purchase
1    6127
0      11
Name: count, dtype: int64

Feature columns: ['age', 'gender', 'device_type', 'time_on_site', 'pages_viewed', 'previous_purchases', 'cart_items', 'discount_seen', 'ad_clicked', 'returning_user', 'avg_session_time', 'bounce_rate']


## 4. Remove redundant feature

`time_on_site` and `avg_session_time` are highly correlated — keep `time_on_site` only.

In [18]:
X = X.drop(columns=["avg_session_time"])
print("Columns after dropping avg_session_time:", list(X.columns))

Columns after dropping avg_session_time: ['age', 'gender', 'device_type', 'time_on_site', 'pages_viewed', 'previous_purchases', 'cart_items', 'discount_seen', 'ad_clicked', 'returning_user', 'bounce_rate']


## 5. Encode categorical variables

In [19]:
X = pd.get_dummies(X, columns=["gender", "device_type"], drop_first=True)
print("Shape after one-hot encoding:", X.shape)
display(X.head())

Shape after one-hot encoding: (6138, 12)


,age,time_on_site,pages_viewed,previous_purchases,cart_items,discount_seen,ad_clicked,returning_user,bounce_rate,gender_Male,device_type_Mobile,device_type_Tablet
1,46.0,15.63,9.0,4.0,6.0,1.0,1.0,1.0,86.73,True,True,False
4,38.0,26.35,9.0,12.0,4.0,1.0,0.0,0.0,55.35,False,True,False
5,56.0,26.60,1.0,12.0,7.0,1.0,1.0,0.0,13.90,False,False,False
6,36.0,7.60,11.0,11.0,6.0,1.0,1.0,1.0,13.12,True,True,False
7,40.0,26.98,7.0,5.0,2.0,1.0,0.0,0.0,12.61,False,True,False


## 6. Outlier handling (IQR → median replacement)

In [20]:
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

def replace_outliers_with_median(frame, columns):
    out = frame.copy()
    for col in columns:
        q1, q3 = out[col].quantile(0.25), out[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        median = out[col].median()
        out[col] = np.where((out[col] < lower) | (out[col] > upper), median, out[col])
    return out

X = replace_outliers_with_median(X, numeric_cols)
print("Outliers replaced with median for:", numeric_cols)

Outliers replaced with median for: ['age', 'time_on_site', 'pages_viewed', 'previous_purchases', 'cart_items', 'discount_seen', 'ad_clicked', 'returning_user', 'bounce_rate']


## 7. Train / validation / test split (stratified)

70% train, 15% validation, 15% test — stratified on `purchase`.

In [21]:
RANDOM_STATE = 42

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

for name, y_split in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test),
]:
    print(f"{name}: {len(y_split)} rows, purchase rate = {y_split.mean():.4f}")

train: 4296 rows, purchase rate = 0.9981
val: 921 rows, purchase rate = 0.9989
test: 921 rows, purchase rate = 0.9978


## 8. Scale features (MinMaxScaler, fit on train only)

In [22]:
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

print("Scaled train sample (first 3 rows):")
display(X_train_scaled.head(3))

Scaled train sample (first 3 rows):


,age,time_on_site,pages_viewed,previous_purchases,cart_items,discount_seen,ad_clicked,returning_user,bounce_rate,gender_Male,device_type_Mobile,device_type_Tablet
3044,0.536585,0.048982,0.388889,0.285714,0.888889,1.0,0.0,1.0,0.308023,1.0,0.0,0.0
5723,0.414634,0.947568,0.222222,0.214286,1.000000,1.0,0.0,1.0,0.334134,0.0,1.0,0.0
4084,0.951220,0.058296,0.888889,0.500000,0.555556,0.0,0.0,0.0,0.553922,0.0,1.0,0.0


## 9. Final checks

In [23]:
assert X_train_scaled.isnull().sum().sum() == 0
assert "user_id" not in X_train_scaled.columns
assert X_train_scaled.shape[1] == X_val_scaled.shape[1] == X_test_scaled.shape[1]

print("Feature count:", X_train_scaled.shape[1])
print("Train shape:", X_train_scaled.shape, "| Val:", X_val_scaled.shape, "| Test:", X_test_scaled.shape)
print("\nClass counts (train / val / test):")
print("  0:", (y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum())
print("  1:", (y_train == 1).sum(), (y_val == 1).sum(), (y_test == 1).sum())

summary = """
Preprocessing complete.
- Use X_train_scaled, y_train for training (tabular / Task A)
- Use X_val_scaled, y_val for hyperparameter tuning
- Use X_test_scaled, y_test for final evaluation only
- Section 10 saves processed/artifacts.pkl (reuse in task_b_sequence_modelling.ipynb)
- Class imbalance: use class weights, stratified metrics, or focus on minority recall
"""
print(summary)

Feature count: 12
Train shape: (4296, 12) | Val: (921, 12) | Test: (921, 12)

Class counts (train / val / test):
  0: 8 1 2
  1: 4288 920 919

Preprocessing complete.
- Use X_train_scaled, y_train for training (tabular / Task A)
- Use X_val_scaled, y_val for hyperparameter tuning
- Use X_test_scaled, y_test for final evaluation only
- Section 10 saves processed/artifacts.pkl (reuse in task_b_sequence_modelling.ipynb)
- Class imbalance: use class weights, stratified metrics, or focus on minority recall



## 10. Save everything to `processed/` 

In [24]:
import json
import pickle
from pathlib import Path

PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)


def build_session_events(row):
    """Synthetic event funnel from one preprocessed session row."""
    events = []
    n_view = max(1, min(int(row["pages_viewed"]), 12))
    events.extend(["view"] * n_view)
    if int(row.get("discount_seen", 0) or 0) == 1:
        events.append("discount")
    if int(row.get("ad_clicked", 0) or 0) == 1:
        events.append("click_ad")
    if int(row.get("cart_items", 0) or 0) > 0:
        events.extend(["cart"] * min(int(row["cart_items"]), 3))
    if int(row["purchase"]) == 1:
        events.append("purchase")
    return events


# --- Event sequences (Task B) ---
sessions = []
for idx, row in df.iterrows():
    ev = build_session_events(row)
    if len(ev) >= 2:
        sessions.append({"session_id": int(idx), "events": ev})

sessions_df = pd.DataFrame(sessions)
sessions_df["length"] = sessions_df["events"].apply(len)

train_idx = set(X_train.index.astype(int))
val_idx = set(X_val.index.astype(int))
test_idx = set(X_test.index.astype(int))

train_sessions = sessions_df[sessions_df["session_id"].isin(train_idx)].copy()
val_sessions = sessions_df[sessions_df["session_id"].isin(val_idx)].copy()
test_sessions = sessions_df[sessions_df["session_id"].isin(test_idx)].copy()

all_events = sorted({e for ev in sessions_df["events"] for e in ev})
PAD_TOKEN = "<pad>"
event2id = {PAD_TOKEN: 0, **{e: i + 1 for i, e in enumerate(all_events)}}
id2event = {v: k for k, v in event2id.items()}

# --- Single bundle for reuse ---
artifacts = {
    "random_state": RANDOM_STATE,
    "df_cleaned": df,
    "splits": {
        "train_idx": [int(i) for i in X_train.index],
        "val_idx": [int(i) for i in X_val.index],
        "test_idx": [int(i) for i in X_test.index],
    },
    # Tabular (Task A / classification)
    "X_train_scaled": X_train_scaled,
    "X_val_scaled": X_val_scaled,
    "X_test_scaled": X_test_scaled,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "scaler": scaler,
    # Sequences (Task B)
    "sessions_df": sessions_df,
    "train_sessions": train_sessions,
    "val_sessions": val_sessions,
    "test_sessions": test_sessions,
    "event2id": event2id,
    "id2event": id2event,
}

with open(PROCESSED_DIR / "artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

# Lightweight copies (optional, human-readable)
df.to_csv(PROCESSED_DIR / "df_cleaned.csv")
with open(PROCESSED_DIR / "split_indices.json", "w") as f:
    json.dump(artifacts["splits"] | {"random_state": RANDOM_STATE}, f, indent=2)

print("Saved processed/artifacts.pkl")
print("  Tabular: X_train_scaled, y_train, scaler, ...")
print("  Sequences: train_sessions, val_sessions, test_sessions, event2id")
print(f"  Sessions with events: {len(sessions_df)} | train/val/test: {len(train_sessions)}/{len(val_sessions)}/{len(test_sessions)}")

Saved processed/artifacts.pkl
  Tabular: X_train_scaled, y_train, scaler, ...
  Sequences: train_sessions, val_sessions, test_sessions, event2id
  Sessions with events: 6138 | train/val/test: 4296/921/921


## 11. Ready for modeling (numpy arrays)

Still in memory; also inside `artifacts.pkl`

In [25]:
# Use in-memory variables, or reload from disk:
# art = pickle.load(open("processed/artifacts.pkl", "rb"))
# X_train_scaled, y_train = art["X_train_scaled"], art["y_train"]

X_train_np = X_train_scaled.values.astype(np.float32)
X_val_np = X_val_scaled.values.astype(np.float32)
X_test_np = X_test_scaled.values.astype(np.float32)
y_train_np = y_train.values.astype(np.float32).reshape(-1, 1)
y_val_np = y_val.values.astype(np.float32).reshape(-1, 1)
y_test_np = y_test.values.astype(np.float32).reshape(-1, 1)

print("X_train:", X_train_np.shape, "| y_train:", y_train_np.shape)

X_train: (4296, 12) | y_train: (4296, 1)
